# Train SEG trực tiếp — Crack và Pothole

**Chạy các cell dưới đây sẽ huấn luyện YOLO thật và in log Epoch / GPU_mem / loss ngay trong notebook.**

1. Tạo môi trường từ `environment.yml`, chọn kernel **Hu hong duong (CUDA)**.
2. Chạy lần lượt: kiểm tra dữ liệu → **3 model × 30 epoch cho mỗi bộ** → xem bảng so sánh.
3. Chạy phần cuối để train **model được chọn × 100 epoch cho mỗi bộ**.

Mỗi lần chạy lại cell cấu hình tạo `RUN_ID` mới, kết quả nằm trong `runs/seg/manual/<RUN_ID>`. Notebook không đọc lại kết quả đã train của các lượt cũ. Không cần mở file HTML.

Trong Anaconda Prompt, tại thư mục project:
```text
conda env create -f environment.yml
conda activate hu-hong-duong
python -m ipykernel install --user --name hu-hong-duong --display-name "Hu hong duong (CUDA)"
```

In [ ]:
from pathlib import Path
from datetime import datetime
import sys, os, time, json, gc, random, math, logging

print("Python đang dùng:", sys.executable)
if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        f"Notebook yêu cầu Python 3.11 nhưng kernel hiện tại dùng {sys.version.split()[0]} "
        f"({sys.executable}). Hãy chọn kernel 'Hu hong duong (CUDA)' rồi Restart Kernel."
    )
os.environ["YOLO_VERBOSE"] = "true"
ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/raw").is_dir() and (p / "models/pretrained").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Không tìm thấy project root. Hãy mở notebook bên trong ..")
SRC = ROOT / "src"
if SRC.is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
(ROOT / "reports/runtime/ultralytics").mkdir(parents=True, exist_ok=True)
os.environ["YOLO_CONFIG_DIR"] = str(ROOT / "reports/runtime/ultralytics")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml, cv2, torch
from PIL import Image
from IPython.display import display
from ultralytics import YOLO
from ultralytics.utils import LOGGER
from ultralytics.models.yolo.segment import SegmentationTrainer
from skimage.morphology import skeletonize

LOGGER.setLevel(logging.INFO)
if not torch.cuda.is_available():
    raise RuntimeError("Chưa nhận CUDA. Kiểm tra kernel hu-hong-duong và bản PyTorch CUDA trong environment.yml.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)

## 1. Cấu hình lượt train mới

Giữ `PILOT_EPOCHS=30`, `MAIN_EPOCHS=100` cho bài chính. `patience=0` tắt dừng sớm để chạy đủ số epoch. Batch nhỏ phù hợp GPU 8 GB.

In [ ]:
DATASETS = {"crack": ROOT / "configs/seg/crack.yaml",
            "pothole": ROOT / "configs/seg/pothole.yaml"}
MODELS = {size: ROOT / "models/pretrained" / f"yolov8{size}-seg.pt"
          for size in ("n", "s", "m")}
BATCH = {"n": 8, "s": 8, "m": 4}
PILOT_EPOCHS, MAIN_EPOCHS = 30, 100
IMGSZ, SEED = 640, 42
ACCURACY_GAP = 0.02  # Chọn nhanh nhất trong nhóm cách mAP50-95 tốt nhất tối đa 2 điểm %
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN_DIR = ROOT / "runs/seg/manual" / RUN_ID
TRAIN_MAIN = True

TRAIN_ARGS = dict(imgsz=IMGSZ, device=0, workers=0, amp=True, cache=False,
                  optimizer="AdamW", lr0=0.001, lrf=0.01,
                  seed=SEED, deterministic=True, patience=0, rect=False,
                  mask_ratio=4, val=True, save=True, save_period=10,
                  plots=True, verbose=True, exist_ok=False)
print("Lượt train mới:", RUN_ID)
print("Nơi lưu weights, results.csv, biểu đồ:", RUN_DIR)

## 2. Kiểm tra dữ liệu cơ bản

Dùng danh sách ảnh đã kiểm tra của project: crack có nhãn `crack`, pothole có nhãn `Pothole`. Cell kiểm tra số lượng, ảnh và polygon mẫu; các ảnh thiếu nhãn đã được loại trong manifest.

In [ ]:
def images_for(dataset, split):
    cfg = yaml.safe_load(DATASETS[dataset].read_text(encoding="utf-8-sig")) or {}
    base_value = Path(str(cfg.get("path", ROOT)))
    base = base_value if base_value.is_absolute() else (DATASETS[dataset].parent / base_value).resolve()
    entry_value = Path(str(cfg[split]))
    entry = entry_value if entry_value.is_absolute() else (base / entry_value).resolve()
    if entry.suffix.lower() != ".txt" or not entry.is_file():
        raise FileNotFoundError(f"Manifest {dataset}/{split} không tồn tại: {entry}")
    paths = [Path(line.strip()) for line in entry.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
    resolved = [p if p.is_absolute() else (entry.parent / p).resolve() for p in paths]
    missing = [p for p in resolved if not p.is_file()]
    if missing:
        raise FileNotFoundError(f"Manifest {dataset}/{split} có ảnh không tồn tại, ví dụ: {missing[0]}")
    return sorted(resolved, key=lambda p:str(p).lower())

def label_for(path):
    parts = list(path.parts)
    parts[max(i for i,x in enumerate(parts) if x.casefold() == "images")] = "labels"
    return Path(*parts).with_suffix(".txt")

rows = []
for dataset, data_yaml in DATASETS.items():
    assert data_yaml.is_file(), data_yaml
    cfg = yaml.safe_load(data_yaml.read_text(encoding="utf-8-sig"))
    print(dataset, "classes:", cfg["names"])
    for split in ("train", "val", "test"):
        images = images_for(dataset,split)
        assert images, f"{dataset}/{split} không có ảnh"
        rows.append({"Dataset":dataset,"Split":split,"Số ảnh":len(images)})
        for path in random.Random(SEED).sample(images,min(20,len(images))):
            with Image.open(path) as picture: picture.verify()
            for line in label_for(path).read_text(encoding="utf-8-sig").splitlines():
                if not line.strip(): continue
                values = np.asarray(line.split(),dtype=float)
                assert len(values)>=7 and (len(values)-1)%2==0, path
                assert values[0].is_integer() and 0<=values[0]<len(cfg["names"]), path
                assert np.isfinite(values).all() and ((values[1:]>=0)&(values[1:]<=1)).all(), path
for weight in MODELS.values(): assert weight.is_file(), weight
display(pd.DataFrame(rows))

## 3. Hàm đánh giá và so sánh

Chọn `best.pt` theo **mask mAP50–95 validation**. Benchmark dùng cùng 200 ảnh validation, batch 1, FP16, 10 lượt warm-up và 3 lần lặp. Thời gian train đo quanh `.train()` gồm cả validation tự động của Ultralytics.

In [ ]:
class MaskTrainer(SegmentationTrainer):
    def validate(self):
        previous_best = self.best_fitness
        metrics, _ = super().validate()
        score = float(metrics["metrics/mAP50-95(M)"])
        self.best_fitness = max(previous_best if previous_best is not None else -math.inf, score)
        return metrics, score

def save_json(path, data):
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(data,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf-8")

def evaluate(best, dataset, output, split="val"):
    model = YOLO(str(best),task="segment")
    result = model.val(data=str(DATASETS[dataset]),split=split,imgsz=IMGSZ,
                       batch=1,device=0,workers=0,half=True,rect=False,
                       conf=.001,iou=.7,max_det=300,plots=True,
                       project=str(output),name=split,exist_ok=True,verbose=True)
    scores = {"P":float(result.seg.mp),"R":float(result.seg.mr),
              "mAP50":float(result.seg.map50),"mAP50-95":float(result.seg.map)}
    assert all(math.isfinite(x) and 0<=x<=1 for x in scores.values())
    del model,result
    gc.collect();torch.cuda.empty_cache()
    return scores

BENCHMARK_PATHS = {}
for dataset in DATASETS:
    paths = images_for(dataset,"val")
    indices = sorted(random.Random(SEED).sample(range(len(paths)),min(200,len(paths))))
    BENCHMARK_PATHS[dataset] = [paths[i] for i in indices]

def benchmark(best,dataset):
    images = [cv2.imread(str(p)) for p in BENCHMARK_PATHS[dataset]]
    assert images and all(x is not None for x in images), f"Không đọc được ảnh benchmark của {dataset}"
    model = YOLO(str(best),task="segment")
    args = dict(imgsz=IMGSZ,device=0,batch=1,half=True,rect=False,conf=.25,
                iou=.7,max_det=300,retina_masks=True,verbose=False,save=False)
    for i in range(10): model.predict(images[i%len(images)],**args)
    elapsed = []
    for repeat in range(3):
        for image in images:
            torch.cuda.synchronize()
            start = time.perf_counter()
            model.predict(image,**args)
            torch.cuda.synchronize()
            elapsed.append((time.perf_counter()-start)*1000)
    mean = float(np.mean(elapsed))
    scores = {"mean_ms":mean,"p95_ms":float(np.percentile(elapsed,95)),"FPS":1000/mean,
              "images":len(images),"repeats":3,"warmup":10}
    del model,images
    gc.collect();torch.cuda.empty_cache()
    return scores

def row_for(record):
    return {"Dataset":record["dataset"],"Model":record["model"],"Epoch":record["epochs"],
            **{k+" (%)":v*100 for k,v in record["validation"].items()},
            "Train (phút)":record["train_seconds"]/60,
            "Suy luận (ms/ảnh)":record["benchmark"]["mean_ms"],"FPS":record["benchmark"]["FPS"]}

## 4. TRAIN THẬT — 3 model × 30 epoch × 2 bộ dữ liệu

**Cell này bắt đầu train khi bạn bấm Run.** Lệnh `model.train(...)` nằm ngay bên dưới; log epoch hiển thị trong output. Thứ tự: crack n/s/m, rồi pothole n/s/m. Để bắt đầu một lượt khác, chạy lại cell cấu hình để có `RUN_ID` mới.

In [ ]:
PILOT_RESULTS = []
for dataset, data_yaml in DATASETS.items():
    for size, pretrained in MODELS.items():
        model_name = f"yolov8{size}-seg"
        project = RUN_DIR / "pilot30" / dataset
        if (project / model_name).exists():
            raise RuntimeError("Lượt này đã có dữ liệu. Tạo RUN_ID mới hoặc dùng cell resume cuối notebook.")
        print(f"\nTRAIN PILOT: {dataset} | {model_name} | {PILOT_EPOCHS} epoch",flush=True)
        model = YOLO(str(pretrained),task="segment")
        started = time.perf_counter()

        # HUẤN LUYỆN THẬT TRÊN GPU — log Epoch/GPU_mem/loss hiển thị ở đây.
        model.train(trainer=MaskTrainer, data=str(data_yaml), epochs=PILOT_EPOCHS,
                    batch=BATCH[size], project=str(project), name=model_name, **TRAIN_ARGS)

        train_seconds = time.perf_counter()-started
        run = Path(model.trainer.save_dir)
        best = run / "weights/best.pt"
        history = pd.read_csv(run / "results.csv")
        assert history["epoch"].astype(int).tolist() == list(range(1,PILOT_EPOCHS+1))
        del model
        gc.collect();torch.cuda.empty_cache()
        record = {"dataset":dataset,"size":size,"model":model_name,"epochs":PILOT_EPOCHS,
                  "best":str(best),"train_seconds":train_seconds,
                  "validation":evaluate(best,dataset,run / "evaluation"),
                  "benchmark":benchmark(best,dataset)}
        PILOT_RESULTS.append(record)
        save_json(run / "evaluation.json",record)
        save_json(RUN_DIR / "pilot_comparison.json",PILOT_RESULTS)
        display(pd.DataFrame([row_for(r) for r in PILOT_RESULTS]).round(3))

## 5. So sánh và chọn model

Chọn riêng cho mỗi bộ: trong nhóm cách mAP50–95 tốt nhất không quá `ACCURACY_GAP`, lấy model có latency thấp nhất; nếu bằng nhau, ưu tiên thời gian train thấp hơn.

In [ ]:
assert len(PILOT_RESULTS)==len(DATASETS)*len(MODELS), "Cần train đủ 6 lượt pilot trước."
COMPARISON = pd.DataFrame([row_for(r) for r in PILOT_RESULTS])
display(COMPARISON.round(3))
SELECTED = {}
for dataset in DATASETS:
    group = [r for r in PILOT_RESULTS if r["dataset"]==dataset]
    assert {r["size"] for r in group}==set(MODELS)
    best_score = max(r["validation"]["mAP50-95"] for r in group)
    candidates = [r for r in group if r["validation"]["mAP50-95"]>=best_score-ACCURACY_GAP-1e-12]
    selected = min(candidates,key=lambda r:(r["benchmark"]["mean_ms"],r["train_seconds"]))
    SELECTED[dataset] = selected["size"]
    print(f"Chọn {dataset}: {selected['model']} | mAP50-95={selected['validation']['mAP50-95']:.4f}")
save_json(RUN_DIR / "selected_models.json",SELECTED)

## 6. TRAIN CHÍNH — model được chọn × đủ 100 epoch

Chỉ chạy cell sau khi xem bảng so sánh. `TRAIN_MAIN=True` cho phép Run All tiếp tục đến đây. Hai lượt train mới bắt đầu từ pretrained; `patience=0` để chạy đủ 100 epoch. Mỗi bài chỉ train một model đã chọn.

In [ ]:
MAIN_RESULTS = []
if TRAIN_MAIN:
    assert set(SELECTED)==set(DATASETS), "Chạy phần chọn model trước."
    for dataset,size in SELECTED.items():
        model_name = f"yolov8{size}-seg"
        project = RUN_DIR / "main100" / dataset
        if (project / model_name).exists():
            raise RuntimeError("Lượt 100 epoch này đã có dữ liệu. Tạo RUN_ID mới hoặc dùng cell resume cuối notebook.")
        print(f"\nTRAIN CHÍNH: {dataset} | {model_name} | {MAIN_EPOCHS} epoch",flush=True)
        model = YOLO(str(MODELS[size]),task="segment")
        started = time.perf_counter()

        # HUẤN LUYỆN THẬT 100 EPOCH — KHÔNG đọc lại trọng số đã train của lượt cũ.
        model.train(trainer=MaskTrainer, data=str(DATASETS[dataset]), epochs=MAIN_EPOCHS,
                    batch=BATCH[size], project=str(project), name=model_name, **TRAIN_ARGS)

        train_seconds = time.perf_counter()-started
        run = Path(model.trainer.save_dir)
        best = run / "weights/best.pt"
        history = pd.read_csv(run / "results.csv")
        assert history["epoch"].astype(int).tolist() == list(range(1,MAIN_EPOCHS+1))
        del model
        gc.collect();torch.cuda.empty_cache()
        record = {"dataset":dataset,"size":size,"model":model_name,"epochs":MAIN_EPOCHS,
                  "best":str(best),"train_seconds":train_seconds,
                  "validation":evaluate(best,dataset,run / "evaluation"),
                  "test":evaluate(best,dataset,run / "evaluation",split="test"),
                  "benchmark":benchmark(best,dataset)}
        MAIN_RESULTS.append(record)
        save_json(run / "evaluation.json",record)
        save_json(RUN_DIR / "main_results.json",MAIN_RESULTS)
    display(pd.DataFrame([{"Dataset":r["dataset"],"Model":r["model"],"Epoch":r["epochs"],
                          **{k+" test (%)":v*100 for k,v in r["test"].items()},
                          "Train (phút)":r["train_seconds"]/60,"FPS":r["benchmark"]["FPS"]}
                         for r in MAIN_RESULTS]).round(3))
else:
    print("Đã chọn chỉ train pilot. Bật TRAIN_MAIN=True khi muốn chạy 100 epoch.")

## 7. Chạy thử SEG và đo pixel

Dùng best.pt vừa train: crack lấy chiều dài skeleton, pothole lấy diện tích hợp mask. Đơn vị px và px².

In [ ]:
for record in MAIN_RESULTS:
    dataset = record["dataset"]
    image_path = images_for(dataset,"val")[0]
    model = YOLO(record["best"],task="segment")
    result = model.predict(str(image_path),imgsz=IMGSZ,device=0,half=True,
                           conf=.25,iou=.7,retina_masks=True,verbose=True)[0]
    mask = np.zeros(result.orig_shape,dtype=bool)
    if result.masks is not None:
        mask = np.any(result.masks.data.cpu().numpy()>.5,axis=0)
        assert mask.shape==result.orig_shape
    overlay = result.plot()
    if dataset=="crack":
        sk = skeletonize(mask)
        h=np.count_nonzero(sk[:,:-1]&sk[:,1:]);v=np.count_nonzero(sk[:-1,:]&sk[1:,:])
        dr=sk[:-1,:-1]&sk[1:,1:]&~sk[:-1,1:]&~sk[1:,:-1]
        dl=sk[:-1,1:]&sk[1:,:-1]&~sk[:-1,:-1]&~sk[1:,1:]
        measurement={"length_px":float(h+v+math.sqrt(2)*(dr.sum()+dl.sum()))}
        overlay[sk]=(0,0,255)
    else:
        measurement={"area_px2":int(mask.sum())}
    print(dataset,measurement)
    plt.figure(figsize=(8,6));plt.imshow(cv2.cvtColor(overlay,cv2.COLOR_BGR2RGB))
    plt.title(f"{dataset}: {measurement}");plt.axis("off");plt.show()
    del model,result
    gc.collect();torch.cuda.empty_cache()

## 8. Tùy chọn: tiếp tục một lượt bị ngắt

Chỉ dùng khi đang train dở. Điền đường dẫn **last.pt của lượt bị ngắt**; để chuỗi rỗng thì cell không chạy. Cell này tiếp tục lịch epoch đã lưu trong checkpoint và hiển thị log trực tiếp. Checkpoint của lượt đã hoàn tất đã bỏ optimizer nên không dùng để resume. Sau khi resume, chạy lại hàm `evaluate()` / `benchmark()` với best.pt để lấy số liệu mới.

In [ ]:
LAST_PT = ""  # Ví dụ: r".\runs\seg\manual\RUN_ID\main100\crack\yolov8n-seg\weights\last.pt"
if LAST_PT:
    path = Path(LAST_PT)
    assert path.is_file(), path
    state = torch.load(path,map_location="cpu",weights_only=False)  # Chỉ checkpoint do bạn train.
    epoch = int(state.get("epoch",-1))
    if epoch<0 or state.get("optimizer") is None:
        raise RuntimeError("Checkpoint này đã hoàn tất hoặc không còn optimizer; không thể resume.")
    total = int(state["train_args"]["epochs"])
    if epoch+1>=total:
        raise RuntimeError("Đã đủ số epoch. Dùng best.pt để đánh giá, không resume nữa.")
    if state["train_args"].get("task")!="segment":
        raise RuntimeError("Cell này dành cho checkpoint SEG.")
    print(f"Tiếp tục sau epoch {epoch+1}, lịch đã lưu: {total} epoch.")
    del state
    model = YOLO(str(path),task="segment")
    model.train(trainer=MaskTrainer,resume=True,device=0,workers=0,verbose=True)

## TRAIN RIÊNG POTHOLE — 150 EPOCH

In [ ]:
# ================================================================
# POTHOLE SEGMENTATION — TRAIN 150 EPOCH
# CELL ĐỘC LẬP HOÀN TOÀN
# ================================================================

from pathlib import Path
from datetime import datetime
import os
import sys
import time
import gc

import torch
from ultralytics import YOLO


# ================================================================
# 1. TÌM PROJECT ROOT
# ================================================================

ROOT = next(
    (
        p.resolve()
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "configs/seg/pothole.yaml").is_file()
    ),
    None
)

if ROOT is None:
    raise RuntimeError(
        "Không tìm thấy project root.\n"
        "Hãy mở notebook bên trong project Hu_hong_duong."
    )

print("=" * 70)
print("PROJECT ROOT:", ROOT)


# ================================================================
# 2. KIỂM TRA CUDA / GPU
# ================================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA chưa hoạt động.\n"
        "Hãy chọn đúng kernel PyTorch CUDA trước khi train."
    )

DEVICE = 0

print("Python :", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("GPU    :", torch.cuda.get_device_name(DEVICE))


# ================================================================
# 3. CẤU HÌNH TRAIN
# ================================================================

DATA_YAML = ROOT / "configs/seg/pothole.yaml"

# ------------------------------------------------
# Chọn model:
# n = nhẹ nhất
# s = cân bằng
# m = mạnh hơn
# ------------------------------------------------

MODEL_SIZE = "m"

PRETRAINED = (
    ROOT
    / "models"
    / "pretrained"
    / f"yolov8{MODEL_SIZE}-seg.pt"
)

EPOCHS = 150
IMGSZ = 640

# GPU 8GB:
# n/s có thể dùng batch 8
# m nên dùng batch 4
BATCH = {
    "n": 8,
    "s": 8,
    "m": 4
}[MODEL_SIZE]

SEED = 42


# ================================================================
# 4. KIỂM TRA FILE
# ================================================================

if not DATA_YAML.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy pothole.yaml:\n{DATA_YAML}"
    )

if not PRETRAINED.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy pretrained model:\n{PRETRAINED}"
    )


# ================================================================
# 5. TẠO THƯ MỤC RUN MỚI
# ================================================================

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

PROJECT_DIR = (
    ROOT
    / "runs"
    / "seg"
    / "pothole150"
    / RUN_ID
)

RUN_NAME = f"yolov8{MODEL_SIZE}-seg_pothole_150ep"


print("=" * 70)
print("POTHOLE SEGMENTATION TRAINING")
print("=" * 70)

print(f"Dataset       : {DATA_YAML}")
print(f"Model         : YOLOv8{MODEL_SIZE}-seg")
print(f"Pretrained    : {PRETRAINED}")
print(f"Epoch         : {EPOCHS}")
print(f"Image size    : {IMGSZ}")
print(f"Batch         : {BATCH}")
print(f"Device        : CUDA:{DEVICE}")
print(f"Optimizer     : AdamW")
print(f"Initial LR    : 0.001")
print(f"Output        : {PROJECT_DIR}")

print("=" * 70)


# ================================================================
# 6. GIẢI PHÓNG GPU TRƯỚC KHI TRAIN
# ================================================================

gc.collect()
torch.cuda.empty_cache()


# ================================================================
# 7. LOAD YOLOv8 SEGMENTATION
# ================================================================

model = YOLO(
    str(PRETRAINED),
    task="segment"
)


# ================================================================
# 8. TRAIN POTHOLE — 150 EPOCH
# ================================================================

start_time = time.perf_counter()

results = model.train(

    # ------------------------
    # DATA
    # ------------------------
    data=str(DATA_YAML),

    # ------------------------
    # TRAIN
    # ------------------------
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,

    # ------------------------
    # GPU
    # ------------------------
    device=DEVICE,
    workers=0,
    amp=True,

    # ------------------------
    # OPTIMIZER
    # ------------------------
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,

    # ------------------------
    # REPRODUCIBILITY
    # ------------------------
    seed=SEED,
    deterministic=True,

    # ------------------------
    # QUAN TRỌNG
    # patience=0:
    # KHÔNG Early Stopping
    # => chạy đủ 150 epoch
    # ------------------------
    patience=0,

    # ------------------------
    # SEGMENTATION
    # ------------------------
    mask_ratio=4,

    # ------------------------
    # VALIDATION
    # ------------------------
    val=True,

    # ------------------------
    # SAVE
    # ------------------------
    save=True,

    # checkpoint mỗi 10 epoch
    save_period=10,

    plots=True,

    # ------------------------
    # DATA CACHE
    # ------------------------
    cache=False,

    rect=False,

    # ------------------------
    # OUTPUT
    # ------------------------
    project=str(PROJECT_DIR),
    name=RUN_NAME,

    exist_ok=False,

    verbose=True
)


# ================================================================
# 9. THỜI GIAN TRAIN
# ================================================================

train_seconds = time.perf_counter() - start_time

print("\n")
print("=" * 70)
print("TRAINING HOÀN THÀNH")
print("=" * 70)

print(
    f"Thời gian train: "
    f"{train_seconds / 60:.2f} phút "
    f"({train_seconds / 3600:.2f} giờ)"
)


# ================================================================
# 10. LẤY ĐƯỜNG DẪN KẾT QUẢ
# ================================================================

RUN_DIR = Path(model.trainer.save_dir)

BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"


print("\nRUN DIRECTORY:")
print(RUN_DIR)

print("\nBEST MODEL:")
print(BEST_PT)

print("\nLAST MODEL:")
print(LAST_PT)

print("\nRESULTS:")
print(RESULTS_CSV)


# ================================================================
# 11. KIỂM TRA MODEL
# ================================================================

assert BEST_PT.is_file(), "Không tìm thấy best.pt"
assert LAST_PT.is_file(), "Không tìm thấy last.pt"
assert RESULTS_CSV.is_file(), "Không tìm thấy results.csv"


# ================================================================
# 12. ĐÁNH GIÁ BEST MODEL TRÊN VALIDATION
# ================================================================

print("\n")
print("=" * 70)
print("VALIDATING BEST MODEL")
print("=" * 70)

best_model = YOLO(
    str(BEST_PT),
    task="segment"
)

val_results = best_model.val(

    data=str(DATA_YAML),

    split="val",

    imgsz=IMGSZ,

    batch=1,

    device=DEVICE,

    workers=0,

    half=True,

    conf=0.001,

    iou=0.7,

    max_det=300,

    plots=True,

    verbose=True
)


# ================================================================
# 13. IN METRICS
# ================================================================

print("\n")
print("=" * 70)
print("KẾT QUẢ POTHOLE SEGMENTATION")
print("=" * 70)

print(
    f"Precision      : "
    f"{val_results.seg.mp:.4f}"
)

print(
    f"Recall         : "
    f"{val_results.seg.mr:.4f}"
)

print(
    f"Mask mAP50     : "
    f"{val_results.seg.map50:.4f}"
)

print(
    f"Mask mAP50-95  : "
    f"{val_results.seg.map:.4f}"
)

print("=" * 70)

print("\nBEST MODEL:")
print(BEST_PT)

print("\n✅ POTHOLE TRAIN 150 EPOCH HOÀN THÀNH")